# Этап 1. Подготовка данных и создание базы знаний
## Банковский консультант: RAG-система

**Стек:** LangChain · pgvector · intfloat/multilingual-e5-large · OpenRouter

## 0. Установка зависимостей

In [ ]:
!pip install langchain langchain-community langchain-huggingface \
             sentence-transformers pgvector psycopg2-binary \
             sqlalchemy python-dotenv nltk tiktoken

## 1. Сбор и подготовка данных

Создаём 5 синтетических документов о банковских продуктах:
1. Потребительские кредиты
2. Ипотека
3. Депозиты и вклады
4. Требования к заёмщикам
5. FAQ — часто задаваемые вопросы

In [ ]:
import re
import json
from datetime import datetime

# ─────────────────────────────────────────────
# Синтетические документы банка «Альфа-Финанс»
# ─────────────────────────────────────────────

raw_documents = [
    {
        "id": "doc_001",
        "title": "Условия потребительского кредитования",
        "product_type": "credit",
        "content": """
        УСЛОВИЯ ПОТРЕБИТЕЛЬСКОГО КРЕДИТОВАНИЯ БАНКА «АЛЬФА-ФИНАНС»

        1. ОБЩИЕ ПОЛОЖЕНИЯ

        Банк «Альфа-Финанс» предоставляет потребительские кредиты физическим лицам на следующих условиях.
        Кредиты выдаются в рублях Российской Федерации.
        Минимальная сумма кредита составляет 50 000 рублей.
        Максимальная сумма кредита составляет 5 000 000 рублей.

        2. ПРОЦЕНТНЫЕ СТАВКИ

        Базовая процентная ставка по потребительскому кредиту составляет от 9,9% годовых.
        Ставка зависит от суммы кредита, срока и кредитной истории заёмщика.
        Для зарплатных клиентов банка предусмотрена скидка 1% к базовой ставке.
        При оформлении страхования жизни ставка снижается на 0,5%.
        Максимальная процентная ставка не превышает 24,9% годовых.

        3. СРОКИ КРЕДИТОВАНИЯ

        Минимальный срок кредита: 6 месяцев.
        Максимальный срок кредита: 84 месяца (7 лет).
        Стандартные сроки: 12, 24, 36, 48, 60, 72, 84 месяца.

        4. ДОСРОЧНОЕ ПОГАШЕНИЕ

        Досрочное погашение кредита разрешено в любой момент без штрафов и комиссий.
        Частичное досрочное погашение возможно от 1 000 рублей.
        При досрочном погашении перерасчёт процентов производится на дату погашения.

        5. НЕОБХОДИМЫЕ ДОКУМЕНТЫ

        Для оформления кредита необходимо предоставить:
        - Паспорт гражданина РФ
        - СНИЛС
        - Справка о доходах (2-НДФЛ или по форме банка) за последние 6 месяцев
        - Для кредитов свыше 1 000 000 рублей: справка с места работы

        6. СПОСОБЫ ПОГАШЕНИЯ

        Кредит можно погашать следующими способами:
        - Через мобильное приложение банка
        - В отделениях банка
        - Через банкоматы и терминалы
        - Автоплатёж с карты или счёта
        - Межбанковский перевод
        """
    },
    {
        "id": "doc_002",
        "title": "Ипотечное кредитование",
        "product_type": "mortgage",
        "content": """
        ИПОТЕЧНОЕ КРЕДИТОВАНИЕ БАНКА «АЛЬФА-ФИНАНС»

        1. ПРОГРАММЫ ИПОТЕКИ

        Банк предлагает следующие ипотечные программы:
        а) Стандартная ипотека — ставка от 11,5% годовых
        б) Льготная ипотека с государственной поддержкой — ставка 8% годовых
        в) Семейная ипотека (для семей с детьми) — ставка 6% годовых
        г) IT-ипотека (для работников IT-отрасли) — ставка 5% годовых
        д) Ипотека на новостройки — ставка от 10,9% годовых

        2. ПАРАМЕТРЫ КРЕДИТОВАНИЯ

        Минимальная сумма ипотеки: 500 000 рублей.
        Максимальная сумма: 30 000 000 рублей (для Москвы и СПб), 15 000 000 рублей для других регионов.
        Срок ипотеки: от 3 до 30 лет.
        Первоначальный взнос: от 15% стоимости объекта.
        По семейной и льготной программам первоначальный взнос от 20%.

        3. ТРЕБОВАНИЯ К ОБЪЕКТУ НЕДВИЖИМОСТИ

        Банк принимает в залог:
        - Квартиры в многоквартирных домах
        - Апартаменты
        - Частные дома и таунхаусы
        - Земельные участки под ИЖС
        Объект должен быть расположен в регионе присутствия банка.
        Не принимаются в залог аварийные дома и дома под снос.

        4. СТРАХОВАНИЕ

        Обязательное страхование предмета залога (имущества) — ежегодно.
        Страхование жизни и здоровья заёмщика — добровольно, но влияет на ставку.
        Без страхования жизни ставка увеличивается на 1,5%.
        Страховая сумма: не менее остатка долга по кредиту.

        5. МАТЕРИНСКИЙ КАПИТАЛ

        Банк принимает материнский капитал в качестве первоначального взноса или для погашения части долга.
        Сумма материнского капитала учитывается при расчёте первоначального взноса.
        Процедура оформления занимает до 10 рабочих дней.

        6. ПРОЦЕСС ОФОРМЛЕНИЯ

        Этапы оформления ипотеки:
        1) Подача заявки онлайн или в отделении — рассмотрение 1-3 дня
        2) Предварительное одобрение — действует 90 дней
        3) Выбор объекта и его оценка (за счёт заёмщика)
        4) Финальное одобрение и подписание договора
        5) Регистрация сделки в Росреестре
        6) Выдача кредитных средств
        """
    },
    {
        "id": "doc_003",
        "title": "Депозиты и вклады",
        "product_type": "deposit",
        "content": """
        ТАРИФЫ И УСЛОВИЯ ПО ВКЛАДАМ БАНКА «АЛЬФА-ФИНАНС»

        1. ВИДЫ ВКЛАДОВ

        ВКЛАД «МАКСИМАЛЬНЫЙ ДОХОД»
        - Ставка: до 16% годовых
        - Срок: 6, 9, 12 месяцев
        - Минимальная сумма: 10 000 рублей
        - Пополнение: не предусмотрено
        - Частичное снятие: не предусмотрено
        - Капитализация: ежемесячная или в конце срока (на выбор)
        - Выплата процентов: в конце срока или ежемесячно на карту

        ВКЛАД «НАКОПИТЕЛЬНЫЙ»
        - Ставка: от 10% до 13% годовых (зависит от суммы и срока)
        - Срок: 3, 6, 12, 24 месяца
        - Минимальная сумма: 1 000 рублей
        - Пополнение: предусмотрено (от 1 000 рублей)
        - Частичное снятие: не предусмотрено
        - Капитализация: ежеквартальная

        ВКЛАД «УНИВЕРСАЛЬНЫЙ»
        - Ставка: от 7% до 9% годовых
        - Срок: бессрочный (до востребования) или фиксированный
        - Минимальная сумма: 1 000 рублей
        - Пополнение: предусмотрено (от 500 рублей)
        - Частичное снятие: предусмотрено с сохранением ставки при остатке от 1 000 рублей
        - Подходит для: накоплений с возможностью снятия

        ВАЛЮТНЫЕ ВКЛАДЫ
        - Валюта: доллары США, евро, юани
        - Ставка в долларах: от 1% до 3% годовых
        - Ставка в евро: от 0,5% до 1,5% годовых
        - Ставка в юанях: от 2% до 4% годовых
        - Минимальная сумма: 500 долларов / 500 евро / 5 000 юаней

        2. СТРАХОВАНИЕ ВКЛАДОВ

        Все вклады физических лиц застрахованы в системе страхования вкладов (ССВ).
        Размер страхового возмещения: до 1 400 000 рублей на одного вкладчика в одном банке.
        Выплата страховки производится АСВ (Агентство по страхованию вкладов) в течение 14 дней.

        3. ДОСРОЧНОЕ РАСТОРЖЕНИЕ

        При досрочном расторжении вклада проценты пересчитываются по ставке «до востребования» (0,01% годовых).
        Исключение: вклад «Универсальный» — ставка сохраняется при соблюдении минимального остатка.

        4. ОТКРЫТИЕ ВКЛАДА

        Вклад можно открыть:
        - Онлайн через мобильное приложение или сайт банка
        - В любом отделении банка
        Для открытия вклада необходим паспорт гражданина РФ.
        Иностранные граждане могут открывать вклады при наличии вида на жительство.
        """
    },
    {
        "id": "doc_004",
        "title": "Требования к заёмщикам",
        "product_type": "requirements",
        "content": """
        ТРЕБОВАНИЯ К ЗАЁМЩИКАМ БАНКА «АЛЬФА-ФИНАНС»

        1. ОБЩИЕ ТРЕБОВАНИЯ

        Для получения любого кредитного продукта заёмщик должен соответствовать следующим требованиям:
        - Гражданство: Российская Федерация
        - Возраст: от 21 года до 70 лет на дату полного погашения кредита
        - Для ипотеки: возраст от 21 до 65 лет на дату последнего платежа
        - Регистрация: постоянная регистрация на территории РФ в регионе присутствия банка

        2. ТРЕБОВАНИЯ К ЗАНЯТОСТИ И ДОХОДУ

        Трудовая занятость:
        - Минимальный стаж на текущем месте работы: 3 месяца
        - Минимальный общий стаж: 1 год
        - Принимаются: наёмные работники, ИП, самозанятые, пенсионеры

        Доход:
        - Минимальный подтверждённый доход: 15 000 рублей в месяц
        - Ежемесячный платёж по кредиту не должен превышать 50% от ежемесячного дохода (ПДН)
        - Для ипотеки рекомендуемый ПДН не более 40%
        - Учитываются: зарплата, пенсия, доход от аренды, дивиденды (с подтверждением)

        Для ИП и самозанятых:
        - Минимальный срок деятельности: 12 месяцев
        - Подтверждение дохода: налоговая декларация за последний год
        - Для самозанятых: справка о доходах из приложения «Мой налог»

        3. ТРЕБОВАНИЯ К КРЕДИТНОЙ ИСТОРИИ

        Банк проверяет кредитную историю через БКИ (бюро кредитных историй).
        Допустимые просрочки: до 30 дней, не более 2 раз за последние 2 года.
        Недопустимо: наличие текущих просрочек более 60 дней.
        Недопустимо: банкротство физического лица в последние 5 лет.
        Скоринговый балл: не ниже 600 (по шкале НБКИ).

        4. СОЗАЁМЩИКИ И ПОРУЧИТЕЛИ

        Привлечение созаёмщиков:
        - Допускается до 3 созаёмщиков
        - Супруг/супруга является обязательным созаёмщиком по ипотеке
        - Доход созаёмщика учитывается при расчёте максимальной суммы кредита

        Поручительство:
        - Поручительство физических лиц принимается по кредитам свыше 500 000 рублей
        - Поручитель должен соответствовать тем же требованиям, что и заёмщик

        5. ПРИЧИНЫ ОТКАЗА В КРЕДИТЕ

        Банк вправе отказать в выдаче кредита без объяснения причин.
        Типичные причины отказа:
        - Низкий кредитный рейтинг
        - Высокая долговая нагрузка (ПДН > 50%)
        - Недостаточный или неподтверждённый доход
        - Наличие текущих просрочек
        - Несоответствие возрастным требованиям
        - Недостоверные сведения в заявке
        """
    },
    {
        "id": "doc_005",
        "title": "Часто задаваемые вопросы (FAQ)",
        "product_type": "faq",
        "content": """
        ЧАСТО ЗАДАВАЕМЫЕ ВОПРОСЫ (FAQ) — БАНК «АЛЬФА-ФИНАНС»

        РАЗДЕЛ 1: КРЕДИТЫ

        В: Как подать заявку на кредит?
        О: Заявку можно подать онлайн на сайте банка, в мобильном приложении или лично в любом отделении банка. Решение по заявке принимается в течение 15 минут — 3 рабочих дней.

        В: Можно ли получить кредит с плохой кредитной историей?
        О: Банк рассматривает каждую заявку индивидуально. При наличии небольших просрочек в прошлом кредит может быть одобрен с более высокой ставкой. Текущие просрочки являются основанием для отказа.

        В: Что такое ПДН и почему это важно?
        О: ПДН (показатель долговой нагрузки) — это отношение ежемесячных платежей по всем кредитам к ежемесячному доходу. По закону банки обязаны учитывать ПДН при выдаче кредитов. При ПДН выше 50% банк, как правило, отказывает в кредите.

        В: Как рассчитать ежемесячный платёж по кредиту?
        О: Воспользуйтесь кредитным калькулятором на сайте банка или в мобильном приложении. Введите сумму, срок и процентную ставку — калькулятор автоматически рассчитает платёж и полную стоимость кредита.

        РАЗДЕЛ 2: ИПОТЕКА

        В: Какой минимальный первоначальный взнос по ипотеке?
        О: Минимальный первоначальный взнос составляет 15% от стоимости объекта по стандартной программе. По льготным программам (семейная, с господдержкой) — от 20%.

        В: Можно ли использовать материнский капитал в качестве первоначального взноса?
        О: Да, банк принимает материнский капитал как первоначальный взнос или для частичного погашения ипотеки. Для этого необходимо предоставить сертификат на материнский капитал и справку из Социального фонда об остатке средств.

        В: Сколько времени занимает оформление ипотеки?
        О: В среднем весь процесс от подачи заявки до получения денег занимает 2–4 недели. Предварительное решение выносится за 1–3 рабочих дня.

        РАЗДЕЛ 3: ВКЛАДЫ

        В: Застрахованы ли мои вклады?
        О: Да. Все вклады физических лиц застрахованы государством в рамках системы страхования вкладов. Максимальная сумма возмещения — 1 400 000 рублей на одного вкладчика в одном банке.

        В: Что будет, если я закрою вклад досрочно?
        О: При досрочном закрытии большинства вкладов проценты пересчитываются по ставке «до востребования» (0,01%). Вклад «Универсальный» позволяет частично снимать средства без потери процентов при сохранении минимального остатка.

        В: Могу ли я открыть вклад онлайн?
        О: Да, вклад можно открыть через мобильное приложение или на сайте банка без посещения отделения. Для этого нужно быть действующим клиентом банка.

        РАЗДЕЛ 4: ОБЩИЕ ВОПРОСЫ

        В: Как связаться со службой поддержки?
        О: Круглосуточная горячая линия: 8-800-123-45-67 (бесплатно по РФ). Чат в мобильном приложении. Email: support@alfa-finance.ru. Отделения работают с 9:00 до 20:00 по будням, с 10:00 до 17:00 в субботу.

        В: Как узнать статус моей заявки?
        О: Статус заявки доступен в личном кабинете на сайте и в мобильном приложении. Также вы получите SMS-уведомление об изменении статуса.

        В: Есть ли у банка мобильное приложение?
        О: Да, приложение «Альфа-Финанс» доступно для iOS и Android. В приложении можно управлять счетами, подавать заявки на кредиты, открывать вклады и получать консультации в чате.
        """
    }
]

print(f"Загружено документов: {len(raw_documents)}")
for doc in raw_documents:
    print(f"  [{doc['id']}] {doc['title']} | тип: {doc['product_type']}")

### 1.1 Очистка и нормализация документов

In [ ]:
def clean_text(text: str) -> str:
    """
    Очистка текста:
    - Удаление лишних пробелов и переносов строк
    - Удаление спецсимволов (кроме стандартной пунктуации)
    - Нормализация дефисов и кавычек
    """
    # Удаляем leading/trailing пробелы
    text = text.strip()
    
    # Заменяем множественные пробелы на один
    text = re.sub(r' {2,}', ' ', text)
    
    # Нормализуем переносы строк: более 2 подряд → 2
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # Убираем пробелы перед знаками препинания
    text = re.sub(r' ([.,;:!?])', r'\1', text)
    
    # Нормализуем кавычки
    text = text.replace('"', '«').replace('"', '»')
    
    # Удаляем непечатаемые символы (кроме \n и \t)
    text = re.sub(r'[^\S\n\t]+', ' ', text)
    
    return text.strip()


def prepare_document(raw_doc: dict) -> dict:
    """Подготовка одного документа к единому формату."""
    return {
        "id": raw_doc["id"],
        "title": raw_doc["title"].strip(),
        "product_type": raw_doc["product_type"],
        "content": clean_text(raw_doc["content"]),
        "char_count": len(clean_text(raw_doc["content"])),
        "created_at": datetime.now().isoformat()
    }


# Применяем очистку ко всем документам
prepared_docs = [prepare_document(doc) for doc in raw_documents]

# Отчёт
for doc in prepared_docs:
    print(f"\n[{doc['id']}] {doc['title']}")
    print(f"  Тип продукта : {doc['product_type']}")
    print(f"  Символов     : {doc['char_count']}")
    print(f"  Превью       : {doc['content'][:100]}...")

## 2. Чанкинг — три стратегии

1. **По размеру** (`CharacterTextSplitter`) — простое разбиение по числу символов
2. **По предложениям** (`NLTKTextSplitter`) — разбиение на уровне предложений
3. **Рекурсивный** (`RecursiveCharacterTextSplitter`) — умное разбиение с учётом структуры

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from langchain.schema import Document
from langchain.text_splitter import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    NLTKTextSplitter,
)

# Преобразуем в формат LangChain Document
lc_documents = [
    Document(
        page_content=doc["content"],
        metadata={
            "doc_id": doc["id"],
            "title": doc["title"],
            "product_type": doc["product_type"],
        }
    )
    for doc in prepared_docs
]

print(f"Документов в формате LangChain: {len(lc_documents)}")

In [ ]:
# Стратегия 1: По размеру (CharacterTextSplitter)
char_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separator="\n"
)
chunks_by_size = char_splitter.split_documents(lc_documents)

# Добавляем метаданные о стратегии и индексе чанка
for i, chunk in enumerate(chunks_by_size):
    chunk.metadata["chunk_strategy"] = "by_size"
    chunk.metadata["chunk_index"] = i

print(f"Стратегия 'По размеру': {len(chunks_by_size)} чанков")
sizes = [len(c.page_content) for c in chunks_by_size]
print(f"  Средний размер: {sum(sizes)//len(sizes)} симв.")
print(f"  Мин / Макс    : {min(sizes)} / {max(sizes)} симв.")

In [ ]:
# Стратегия 2: По предложениям (NLTKTextSplitter)
nltk_splitter = NLTKTextSplitter(
    chunk_size=600,
    chunk_overlap=60,
    language="russian"
)
chunks_by_sentence = nltk_splitter.split_documents(lc_documents)

for i, chunk in enumerate(chunks_by_sentence):
    chunk.metadata["chunk_strategy"] = "by_sentence"
    chunk.metadata["chunk_index"] = i

print(f"Стратегия 'По предложениям': {len(chunks_by_sentence)} чанков")
sizes = [len(c.page_content) for c in chunks_by_sentence]
print(f"  Средний размер: {sum(sizes)//len(sizes)} симв.")
print(f"  Мин / Макс    : {min(sizes)} / {max(sizes)} симв.")

In [ ]:
# Стратегия 3: Рекурсивный (RecursiveCharacterTextSplitter)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)
chunks_recursive = recursive_splitter.split_documents(lc_documents)

for i, chunk in enumerate(chunks_recursive):
    chunk.metadata["chunk_strategy"] = "recursive"
    chunk.metadata["chunk_index"] = i

print(f"Стратегия 'Рекурсивная': {len(chunks_recursive)} чанков")
sizes = [len(c.page_content) for c in chunks_recursive]
print(f"  Средний размер: {sum(sizes)//len(sizes)} симв.")
print(f"  Мин / Макс    : {min(sizes)} / {max(sizes)} симв.")

In [ ]:
# Сравнение стратегий
print(f"{'Стратегия':<25} {'Чанков':>8} {'Ср. размер':>12} {'Мин':>8} {'Макс':>8}")

strategies = [
    ("По размеру",     chunks_by_size),
    ("По предложениям", chunks_by_sentence),
    ("Рекурсивная",    chunks_recursive),
]

for name, chunks in strategies:
    sizes = [len(c.page_content) for c in chunks]
    print(f"{name:<25} {len(chunks):>8} {sum(sizes)//len(sizes):>12} {min(sizes):>8} {max(sizes):>8}")


print("вывод:")
print("  - 'По размеру'     : быстро, но режет на середине мысли/предложения.")
print("  - 'По предложениям': сохраняет семантику, но чанки неодинаковы.")
print("  - 'Рекурсивная'    : оптимальный баланс — выбрана для эмбеддингов.")

# Финальный набор чанков для дальнейшей работы
final_chunks = chunks_recursive
print(f"\nИтого чанков для эмбеддинга: {len(final_chunks)}")

In [ ]:
# Просмотр примера чанка
print("Пример чанка (рекурсивная стратегия)")
sample = final_chunks[5]
print(f"Документ    : {sample.metadata['title']}")
print(f"Тип продукта: {sample.metadata['product_type']}")
print(f"Стратегия   : {sample.metadata['chunk_strategy']}")
print(f"Размер      : {len(sample.page_content)} симв.")
print(f"Текст       :\n{sample.page_content}")

## 3. Создание эмбеддингов

### Выбор модели

**Выбрана модель:** `intfloat/multilingual-e5-large`

In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings

# Инициализация модели эмбеддингов
embeddings_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cpu"},   # замените на "cuda" если есть GPU
    encode_kwargs={
        "normalize_embeddings": True,  # косинусное сходство работает корректнее
        "batch_size": 32,
    },
)

# Проверяем размерность
test_vec = embeddings_model.embed_query("тестовый запрос")
print(f"Модель загружена. Размерность вектора: {len(test_vec)}")

## 4. Сохранение в pgvector

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Загружаем .env если есть

# Настройки подключения к PostgreSQL
# свои параметры (или в .env)
PG_HOST     = os.getenv("PG_HOST",     "localhost")
PG_PORT     = os.getenv("PG_PORT",     "5432")
PG_DB       = os.getenv("PG_DB",       "bank_rag")
PG_USER     = os.getenv("PG_USER",     "postgres")
PG_PASSWORD = os.getenv("PG_PASSWORD", "admin")

CONNECTION_STRING = (
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}"
    f"@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

COLLECTION_NAME = "bank_documents"
print(f"Строка подключения: postgresql://***@{PG_HOST}:{PG_PORT}/{PG_DB}")

In [ ]:
from langchain_community.vectorstores import PGVector

# Создаём векторное хранилище и сохраняем все чанки
# pre_delete_collection=True — пересоздаёт таблицу при повторном запуске
vectorstore = PGVector.from_documents(
    documents=final_chunks,
    embedding=embeddings_model,
    collection_name=COLLECTION_NAME,
    connection_string=CONNECTION_STRING,
    pre_delete_collection=True,
)

print(f"Успешно сохранено {len(final_chunks)} чанков в pgvector")
print(f"Коллекция: {COLLECTION_NAME}")

In [ ]:
# Проверка: тестовый поиск
test_query = "Какова минимальная сумма кредита?"
results = vectorstore.similarity_search(test_query, k=3)

print(f"Тестовый запрос: '{test_query}'")
for i, doc in enumerate(results, 1):
    print(f"\n[Результат {i}]")
    print(f"  Документ  : {doc.metadata['title']}")
    print(f"  Тип       : {doc.metadata['product_type']}")
    print(f"  Фрагмент  : {doc.page_content[:200]}...")

In [ ]:
# Сохраняем конфигурацию для следующих этапов
config = {
    "connection_string": CONNECTION_STRING,
    "collection_name": COLLECTION_NAME,
    "embedding_model": "intfloat/multilingual-e5-large",
    "chunk_strategy": "recursive",
    "chunk_size": 500,
    "chunk_overlap": 50,
    "total_chunks": len(final_chunks),
    "total_documents": len(prepared_docs),
}

with open("rag_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Конфигурация сохранена в rag_config.json")
print(json.dumps(config, ensure_ascii=False, indent=2))

**Следующий шаг → Этап 2: Система ретрива**